# Google Colab Setup - Melanoma Classification

Ovaj notebook priprema okruzenje na Google Colab-u:
1. Instalacija zavisnosti
2. Preuzimanje dataseta sa Kaggle-a
3. Povezivanje sa Google Drive-om za cuvanje rezultata
4. Verifikacija da sve radi

**Pokrenite ovaj notebook JEDNOM pre ostalih Colab notebookova.**

## 0. Provera GPU-a

Proverite da imate GPU runtime: Runtime -> Change runtime type -> GPU

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA dostupna: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memorija: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("UPOZORENJE: GPU nije dostupan! Idite na Runtime -> Change runtime type -> GPU")

## 1. Instalacija zavisnosti

In [ ]:
!pip install -q timm albumentations mahotas

## 2. Kloniranje repozitorijuma

Kloniramo nas repozitorijum sa GitHub-a da bismo imali `src/` module.

In [ ]:
import os

REPO_URL = "https://github.com/VASE-IME/melanoma-classification.git"  # <-- PROMENITE na vas URL
REPO_DIR = "/content/melanoma-classification"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repo vec postoji na {REPO_DIR}")

# Dodaj src/ na Python path
import sys
sys.path.insert(0, REPO_DIR)

print(f"src/ moduli dostupni: {os.path.exists(os.path.join(REPO_DIR, 'src', 'config.py'))}")

## 3. Kaggle API Setup

Da biste skinuli dataset sa Kaggle-a, potreban vam je `kaggle.json` fajl:
1. Idite na https://www.kaggle.com/settings
2. Kliknite "Create New Token" u sekciji API
3. Preuzmite `kaggle.json` fajl
4. Upload-ujte ga kada vas sledeca celija zatrazi

In [ ]:
# Upload kaggle.json
from google.colab import files

if not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
    print("Upload-ujte vas kaggle.json fajl:")
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    !mv kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("Kaggle API konfigurisan!")
else:
    print("Kaggle API vec konfigurisan.")

## 4. Preuzimanje dataseta sa Kaggle-a

Dataset: [ISIC Challenge Dataset 2020](https://www.kaggle.com/datasets/sumaiyabinteshahid/isic-challenge-dataset-2020)

**Napomena:** Preuzimanje i raspakivanje moze da potraje ~10-15 minuta (23GB).
Dataset se cuva na **lokalnom disku Colab VM-a** (~100GB dostupno), ne na Google Drive-u.

In [ ]:
DATA_DIR = "/content/data"

if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR, exist_ok=True)
    print("Preuzimanje dataseta sa Kaggle-a (~23GB)...")
    !kaggle datasets download -d sumaiyabinteshahid/isic-challenge-dataset-2020 -p {DATA_DIR}
    print("\nRaspakivanje...")
    !unzip -q {DATA_DIR}/*.zip -d {DATA_DIR}
    !rm -f {DATA_DIR}/*.zip  # Oslobodi prostor
    print("Gotovo!")
else:
    print(f"Dataset vec postoji na {DATA_DIR}")

## 5. Pronalazenje strukture dataseta

Proveravamo gde su slike i CSV fajlovi nakon raspakivanja.

In [ ]:
import glob

# Pronadji train.csv
csv_files = glob.glob(f"{DATA_DIR}/**/train.csv", recursive=True)
print("Pronadjeni CSV fajlovi:")
for f in csv_files:
    print(f"  {f}")

# Pronadji folder sa slikama
jpg_files = glob.glob(f"{DATA_DIR}/**/*.jpg", recursive=True)
print(f"\nUkupno JPG slika: {len(jpg_files)}")

if jpg_files:
    # Odredi folder sa slikama
    img_dir = os.path.dirname(jpg_files[0])
    print(f"Folder sa slikama: {img_dir}")
    print(f"Primer slike: {os.path.basename(jpg_files[0])}")

In [ ]:
# Automatsko prepoznavanje putanja
import pandas as pd

TRAIN_CSV = csv_files[0] if csv_files else None
TRAIN_DIR = img_dir if jpg_files else None

print(f"TRAIN_CSV = {TRAIN_CSV}")
print(f"TRAIN_DIR = {TRAIN_DIR}")

if TRAIN_CSV:
    df = pd.read_csv(TRAIN_CSV)
    print(f"\nBroj redova u CSV: {len(df)}")
    print(f"Kolone: {list(df.columns)}")
    print(f"\nRaspodela klasa:")
    print(f"  Benign:    {(df['target'] == 0).sum()} ({(df['target'] == 0).mean()*100:.1f}%)")
    print(f"  Malignant: {(df['target'] == 1).sum()} ({(df['target'] == 1).mean()*100:.1f}%)")
    print(f"\nBroj pacijenata: {df['patient_id'].nunique()}")

## 6. Google Drive za cuvanje rezultata

Rezultate cuvamo na Google Drive-u da ne nestanu kada se Colab sesija zavrsi.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RESULTS_DIR = '/content/drive/MyDrive/melanoma_results'
MODELS_DIR = f'{RESULTS_DIR}/models'
os.makedirs(MODELS_DIR, exist_ok=True)
print(f"Rezultati ce se cuvati u: {RESULTS_DIR}")

## 7. Test konfiguracije

In [ ]:
from src.config import Config

cfg = Config.colab()

# Azuriraj putanje na osnovu pronadjene strukture
cfg.train_csv = TRAIN_CSV
cfg.train_dir = TRAIN_DIR
cfg.model_save_dir = MODELS_DIR

print(f"Model: {cfg.model_type}")
print(f"Device: {cfg.get_device()}")
print(f"Epochs: {cfg.epochs}")
print(f"Folds: {cfg.num_folds}")
print(f"Batch size: {cfg.batch_size}")
print(f"Train CSV: {cfg.train_csv}")
print(f"Train DIR: {cfg.train_dir}")
print(f"Tone mapping: {cfg.tone_mapping_csv}")
print(f"Models DIR: {cfg.model_save_dir}")

In [ ]:
# Verifikacija - probaj ucitati jednu sliku
import cv2
import matplotlib.pyplot as plt

sample_img_name = df.iloc[0]['image_name']
sample_path = os.path.join(TRAIN_DIR, f"{sample_img_name}.jpg")

if os.path.exists(sample_path):
    img = cv2.imread(sample_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(5, 5))
    plt.imshow(img_rgb)
    plt.title(f"{sample_img_name} - target: {df.iloc[0]['target']}")
    plt.axis('off')
    plt.show()
    print(f"Slika ucitana uspesno! Dimenzije: {img.shape}")
else:
    print(f"GRESKA: Slika ne postoji na putanji: {sample_path}")
    print("Proverite TRAIN_DIR putanju iznad.")

## 8. Sacuvajte putanje za ostale notebookove

Sacuvajte ove putanje - bice vam potrebne u svakom sledecem notebook-u.

In [ ]:
# Sacuvaj putanje u fajl za ostale notebookove
import json

paths = {
    'train_csv': TRAIN_CSV,
    'train_dir': TRAIN_DIR,
    'results_dir': RESULTS_DIR,
    'models_dir': MODELS_DIR,
    'repo_dir': REPO_DIR,
}

with open('/content/colab_paths.json', 'w') as f:
    json.dump(paths, f, indent=2)

# Takodje sacuvaj na Drive za persistenciju
with open(f'{RESULTS_DIR}/colab_paths.json', 'w') as f:
    json.dump(paths, f, indent=2)

print("Putanje sacuvane! Mozete nastaviti sa ostalim Colab notebookovima.")
print("\nU svakom sledecem notebook-u, na pocetku pokrenite:")
print('  import json')
print('  paths = json.load(open("/content/colab_paths.json"))')

## Zakljucak

Okruzenje je spremno! Sada mozete pokrenuti:
- `colab_training_cnn.ipynb` - treniranje Custom CNN modela
- `colab_training_efficientnet.ipynb` - treniranje EfficientNet modela
- `colab_evaluation.ipynb` - poredjenje modela
- `colab_fairness.ipynb` - analiza pravicnosti

**Napomena:** Ako vam se Colab sesija prekine, ponovo pokrenite ovaj notebook (korak 4 ce preskociti preuzimanje ako je vec zavrseno u istoj sesiji).